In [44]:
import json, time, random, threading, math
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
from sentence_transformers import SentenceTransformer
import requests

In [45]:
FEED = Path("live_feed.jsonl")

def append(rec):
    with FEED.open("a") as f:
        f.write(json.dumps(rec) + "\n")

def wiki_feed(max_seconds=None):
    url = "https://stream.wikimedia.org/v2/stream/recentchange"
    headers = {"User-Agent": "rag-prototype/0.1 (trivedi.utkarsh2934@gmail.com)"}
    start = time.time()
    with requests.get(url, stream=True, headers=headers, timeout=60) as r:
        print("HTTP status:", r.status_code)
        r.raise_for_status()
        for line in r.iter_lines():
            if max_seconds and time.time() - start > max_seconds:
                break
            if not line.startswith(b"data: "):
                continue
            try:
                e = json.loads(line[6:])
            except json.JSONDecodeError:
                continue
            if e.get("wiki") != "enwiki" or e.get("type") != "edit":
                continue
            append({
                "id": str(e["meta"]["id"]),
                "ts": datetime.fromtimestamp(e["timestamp"], timezone.utc).isoformat(),
                "text": f"{e['user']} edited '{e['title']}': {e.get('comment', '')}",
            })

wiki_feed(max_seconds=15)
print(FEED.stat().st_size, "bytes")
print(FEED.read_text().splitlines()[-3:])

HTTP status: 200
311727 bytes
['{"id": "4e2ca128-905a-4bfb-8fde-f6c889315646", "ts": "2026-10-10T20:46:17+00:00", "text": "Benboy250 edited \'Seafarers\' Pensions Convention, 1946\': +[[Category:Maritime law treaties of Argentina]]; Maritime law treaties by country ([[toolforge:quickcategories/batch/18647/|QuickCategories batch #18647]])"}', '{"id": "evt-680", "ts": "2026-10-10T20:46:18.203997+00:00", "text": "Payment gateway latency spiked to 900ms for 3 minutes (seq 680)"}', '{"id": "413124b7-bfd8-4a25-a6b1-a54923a29f5c", "ts": "2026-10-10T20:46:13+00:00", "text": "Likeanechointheforest edited \'Angela Anaconda\': Reverted [[WP:AGF|good faith]] edits by [[Special:Contributions/~2026-54509-00|~2026-54509-00]] ([[User talk:~2026-54509-00|talk]]): needs source ([[WP:AntiVandal|AV]])"}']


In [46]:
FEED = Path("live_feed.jsonl")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

def simulate_feed():
    samples = [
        "Warehouse A reports stock of SKU-1042 dropped below reorder level",
        "Promo PR-77 uplift is tracking 12% above forecast in the north region",
        "Pipeline gold_sales finished in 14 minutes, 2 late-arriving files",
        "Payment gateway latency spiked to 900ms for 3 minutes",
    ]
    i = 0
    while True:
        rec = {"id": f"evt-{i}",
               "ts": datetime.now(timezone.utc).isoformat(),
               "text": random.choice(samples) + f" (seq {i})"}
        with FEED.open("a") as f:
            f.write(json.dumps(rec) + "\n")
        i += 1
        time.sleep(random.uniform(1, 3))

threading.Thread(target=simulate_feed, daemon=True).start()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [47]:
# Cell 3: incremental index that tails the file
class LiveIndex:
    def __init__(self, path):
        self.path, self.offset = path, 0
        self.docs, self.seen = [], set()
        self.vecs = np.empty((0, embedder.get_sentence_embedding_dimension()), dtype="float32")

    def refresh(self):
        """Read only the new complete lines since the last call, embed and add them."""
        if not self.path.exists():
            return 0
        if self.path.stat().st_size < self.offset:   # file truncated or rotated
            self.offset = 0
        with self.path.open("rb") as f:
            f.seek(self.offset)
            chunk = f.read()
        end = chunk.rfind(b"\n")                     # ignore a half-written last line
        if end == -1:
            return 0
        self.offset += end + 1

        new = []
        for line in chunk[:end].decode().splitlines():
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                continue
            if r["id"] not in self.seen:             # idempotent on id
                self.seen.add(r["id"])
                new.append(r)
        if new:
            v = embedder.encode([r["text"] for r in new], normalize_embeddings=True)
            self.vecs = np.vstack([self.vecs, v.astype("float32")])
            self.docs.extend(new)
        return len(new)

    def search(self, query, k=5, half_life_s=600, recency_weight=0.3):
        if not self.docs:
            return []
        q = embedder.encode([query], normalize_embeddings=True)[0]
        sim = self.vecs @ q
        now = datetime.now(timezone.utc)
        age = np.array([(now - datetime.fromisoformat(d["ts"])).total_seconds() for d in self.docs])
        score = sim + recency_weight * np.exp(-age * math.log(2) / half_life_s)
        top = np.argsort(-score)[:k]
        return [(self.docs[i], float(sim[i])) for i in top]

index = LiveIndex(FEED)

/tmp/ipykernel_6673/266741064.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.vecs = np.empty((0, embedder.get_sentence_embedding_dimension()), dtype="float32")


In [48]:
from google.colab import userdata
from google import genai
from google.genai import types
from google.genai import errors


gclient = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

MODELS = ["gemini-3.8-flash"]   # add a fallback from gclient.models.list() if you want one

def generate_with_retry(contents, system_instruction, max_attempts=5):
    last_err = None
    for attempt in range(max_attempts):
        for model in MODELS:
            try:
                resp = gclient.models.generate_content(
                    model=model,
                    contents=contents,
                    config=types.GenerateContentConfig(system_instruction=system_instruction),
                )
                return resp.text
            except errors.ServerError as e:                       # 5xx: overloaded or unavailable
                last_err = e
            except errors.ClientError as e:
                if e.code == 429:                                 # rate limited, worth retrying
                    last_err = e
                else:                                             # 400/401/404: retrying won't help
                    raise
        wait = min(2 ** attempt, 20) + random.random()            # 1s, 2s, 4s, 8s... plus jitter
        print(f"Gemini busy ({last_err.code}), retrying in {wait:.1f}s...")
        time.sleep(wait)
    raise last_err

def feed_status():
    age = datetime.now(timezone.utc).timestamp() - FEED.stat().st_mtime
    print(f"feed last written {age:.0f}s ago | {FEED.stat().st_size/1e6:.1f} MB")

def expand_query(question):
    txt = generate_with_retry(
        contents=("Give 10-15 short search terms, synonyms and related entities (leagues, "
                  "title suffixes like 'F.C.', roles like 'striker') that help find Wikipedia "
                  "edit events relevant to this question. Return one comma-separated line only.\n"
                  f"Question: {question}"),
        system_instruction="You expand search queries.",
    )
    return [t.strip() for t in txt.split(",") if t.strip()]

def retrieve(question, terms, k=40, recency_weight=0.1, half_life_s=1800):
    Q = embedder.encode([question, " ".join(terms)], normalize_embeddings=True)
    sim = (index.vecs @ Q.T).max(axis=1)                       # best match to either query
    low = [t.lower() for t in terms]
    kw = np.array([0.1 * any(t in d["text"].lower() for t in low) for d in index.docs])
    now = datetime.now(timezone.utc)
    age = np.array([(now - datetime.fromisoformat(d["ts"])).total_seconds() for d in index.docs])
    score = sim + kw + recency_weight * np.exp(-age * math.log(2) / half_life_s)
    top = np.argsort(-score)[:k]
    return [(index.docs[i], float(score[i]), float(age[i])) for i in top]

def fmt_age(s):
    return f"{int(s)}s ago" if s < 120 else f"{int(s // 60)} min ago" if s < 7200 else f"{s / 3600:.1f} h ago"

SYSTEM = """You answer questions about a live feed of Wikipedia edit events.
Rules:
1. The events are the only source for what was edited and when.
2. Decide whether a page relates to the topic from its title, edit summary, and your general
   knowledge (e.g. a person's name may be a footballer). Mark those cases "(inferred)".
3. Ignore events that are not relevant. Do not list them.
4. Format: one intro line with the count, then one bullet per page:
   **Page title** - what was edited (section/summary), age, event id.
   Then one final line: the most recent relevant edit and its age.
5. If nothing is relevant, say so and name the closest events instead.
Use the age values given; do not compute times yourself."""

def ask(question, k=40, debug=False):
    index.refresh()
    terms = expand_query(question)
    hits = retrieve(question, terms, k=k)
    if debug:
        print("terms:", terms)
        for d, sc, a in hits[:15]:
            print(f"  {sc:.2f} | {fmt_age(a):>10} | {d['text'][:80]}")
    context = "\n".join(f"[{d['id']} | {fmt_age(a)}] {d['text']}" for d, _, a in hits)
    return generate_with_retry(
        contents=f"Events:\n{context}\n\nQuestion: {question}",
        system_instruction=SYSTEM,
    )


In [49]:
print(ask("Which pages are being edited about football or soccer?", debug=True))

Gemini busy (503), retrying in 1.4s...
terms: ['football', 'soccer', 'F.C.', 'association football', 'footballer', 'FIFA', 'UEFA', 'Premier League', 'striker', 'goalkeeper', 'midfielder', 'football manager', 'FIFA World Cup', 'Champions League']
  0.67 |  4 min ago | Txikon edited 'Spain women's national football team results': /* Stats */
  0.59 |  4 min ago | RevampedEditor edited 'João Carvalho (footballer, born 1997)': Undid revision [[
  0.58 |  4 min ago | ~2026-54511-00 edited 'Armenia women's national football team': update
  0.54 | 24 min ago | Mediocre Legacy edited 'United Arab Emirates national football team': /* Arab Cu
  0.51 | 40 min ago | Ticatsfan1999 edited 'Sports in the San Francisco Bay Area': /* Major profession
  0.49 |    95s ago | Spitzmauskc edited 'User:Spitzmauskc/ice hockey team': /* Players and personnel 
  0.49 | 24 min ago | ~2026-49098-70 edited '2026–27 Stockport County F.C. season': /* Matches */
  0.47 |  4 min ago | The Editor 155 edited 'Hallowarri